# Intertextuality Evaluator

**The Intertextuality Evaluator** assesses how much a text depends on readers bringing outside knowledge to it for students in grades 3-11. It considers not content knowledge, but relational knowledge: familiarity with other texts, cultural references, genre conventions, and shared discourse that the text assumes without explaining. When you run a passage through the evaluator, it returns a structured output that includes:

* **complexity_score**: The Intertextuality complexity level (Slightly to Exceedingly Complex).
* **adjustment_and_scaffolding**: Analyzing what the author assumes the reader knows vs what is explained.
* **detailed_summary**: Individual complexity factors that drive the rating, with descriptions and their effect on the dimension.
* **recommended_use_cases**: Additional instructional opportunities for using the text.
* **reasoning**: A synthesis of why the text fits the chosen complexity level.

This gives you a clear signal about the intertextuality demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic textstat typing_extensions

In [ ]:
import getpass
import os
from dotenv import load_dotenv
import json
import hashlib
from pathlib import Path
from typing import List
from enum import Enum
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import textstat
from IPython.display import Markdown
import pprint as pp

In [3]:
# Check for the API key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [4]:
ASSETS_DIR = Path(".")
config_path = ASSETS_DIR / "config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

# Load standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config (system, user, ...). Each
# message has {role, source_path, sha256}. We verify each file's sha256
# matches the declared hash -- drift tripwire #1, applied to every prompt
# regardless of role. CI should promote a mismatch to a hard failure.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    path = ASSETS_DIR / msg_spec["source_path"]
    text = path.read_text()
    actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

print(
    f"Loaded {CONFIG['evaluator']['id']} "
    f"from {ASSETS_DIR.resolve()}"
)
print(f"  model:       {CONFIG['steps'][0]['model']['name']}")
print(f"  temperature: {CONFIG['steps'][0]['generation']['temperature']}")
print(f"  prompts:")
for msg_spec, (role, text) in zip(CONFIG["steps"][0]["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<14} ({len(text):>5} chars, sha {sha})")

Loaded literacy.gla.intertextuality from /Users/achi/Documents/evaluators/evals/literacy/qualitative-text-complexity/intertextuality
  model:       gemini-3-flash-preview
  temperature: 0.0
  prompts:
    system  system.txt     ( 3849 chars, sha 30302fe959ad)
      user  user.txt       (   63 chars, sha cd8e6347db1a)


In [5]:
# -------------------------------------------------------------------------
# FK score helper (declared as a preprocessing step in CONFIG['preprocessing'])
# -------------------------------------------------------------------------
_FK_PRE = next(p for p in CONFIG["preprocessing"] if p["id"] == "fk_score")
_FK_IMPL = _FK_PRE["implementation"]["python"]
_FK_LIB = _FK_IMPL["library"]
_FK_FN = _FK_IMPL["function"]
_FK_TRANSFORM = _FK_IMPL["post_transform"]

if _FK_LIB != "textstat":
    raise ValueError(f"unsupported fk library in config: {_FK_LIB!r}")


def calculate_fk_score(text) -> float:
    """Compute Flesch-Kincaid Grade Level per CONFIG['preprocessing']."""
    fn = getattr(textstat, _FK_FN)
    value = fn(text)
    if _FK_TRANSFORM["type"] == "round":
        value = round(value, _FK_TRANSFORM["precision"])
    else:
        raise ValueError(f"unsupported post_transform type: {_FK_TRANSFORM['type']!r}")
    return value


# -------------------------------------------------------------------------
# Evaluator function: model / prompt / parser config all read from CONFIG
# -------------------------------------------------------------------------
_STEP = CONFIG["steps"][0]  # single-step evaluator today. Extensible to multi-step evaluators.

def evaluate_text_complexity(text: str, grade_level: int):
    """
    Evaluate the Intertextuality-dimension complexity of a text using the canonical
    config in evals/literacy/qualitative-text-complexity/intertextuality/config.json + system.txt + user.txt.

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model
                          (input-side trace).
      - raw_output:       the AIMessage object returned by the LLM
                          (preserves response_metadata, usage_metadata).
      - raw_text:         just the string content of the AIMessage.
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA.
      - usage:            token-usage metadata if the provider returned it.

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage
    and the parsed output without a second call.
    """
    # 1. Structured output -- parser.kind == "structured_output" uses the model's
    #    native output enforcement. OUTPUT_SCHEMA is loaded from output_schema.json,
    #    the standalone source of truth. include_raw=True preserves the AIMessage
    #    for tracing alongside the parsed result.
    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    # 2. Prompt template -- every message's content was loaded from disk
    #    and verified against config in the loader cell. We just feed the
    #    (role, text) tuples straight into ChatPromptTemplate.
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        # Step A: Calculate FK Score
        fk_score = calculate_fk_score(text)
        print(f"Calculated Flesch-Kincaid Score: {fk_score}")

        inputs = {"text": text, "grade_level": grade_level, "fk_score": fk_score}

        # Step B: Render the prompt up-front so we can return exactly what
        #         was sent to the model (input-side trace).
        rendered_messages = prompt_template.format_messages(**inputs)

        # Step C: Single LLM call -> raw AIMessage + parsed output dict.
        #         No second LLM call.
        raw = structured_llm.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        # Step D: Return the full trace dict.
        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output":       raw["raw"],
            "raw_text":         raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage":            getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

In [6]:
sample_text = """
"Why Is This Program Called Artemis?\nThe first astronauts landed on the Moon in 1969. The missions were called Apollo. The name Apollo came from stories told by Greek people long ago. In the stories, Apollo was a god. Apollo had a twin sister. Her name was Artemis. She was the goddess of the Moon in the Greek stories.\n\nWhat Spacecraft Will Be Used for the Artemis Program?\nNASA has a new rocket. It is the Space Launch System. It is called SLS for short. It is the most powerful rocket in the world. SLS will carry the Orion spacecraft on top. Orion can carry up to four astronauts. Orion will fly around, or orbit, the Moon. The crew will take trips in spacecraft called landers to get to work on the surface of the Moon. When all of their work is finished, the crew will return to Earth aboard Orion.\n\nWhen Will Artemis Go to the Moon?\nThe first Apollo missions were tests. NASA launched the rocket to be sure it was safe for people and work as planned. Artemis will be tested first, too: Artemis 1 launched SLS and Orion with no astronauts on Nov. 16, 2022. Artemis 2 is carrying astronauts. They will circle past the Moon and return to Earth. Artemis 3 will send a crew with the next man to land on the Moon. Artemis 4 will send astronauts to land on the Moon.\n\nWhat Will Artemis Astronauts Do on the Moon?\nThe Artemis 4 crew will visit the Moon’s South Pole. No one has ever been there. At the Moon, astronauts will:\nSearch for the Moon’s water and use it.\nLearn how to live and work on a different planet or moon.\nTest the new tools that astronauts will need for a mission to Mars.\n\nWhy Is the Artemis Program Important?\nThe Moon is a good place to learn new science. NASA will learn more about the Moon, Earth, and even the Sun. The Moon is also a place to learn how astronauts can one day live and work on Mars.\nAstronauts on the Artemis missions will need new tools. Many companies will make these new tools. This will mean new jobs for people and companies on Earth. Other countries will be NASA’s partners for the new Moon missions. They will work on Artemis to bring the world together for a mission to Earth’s nearest neighbor in space."
"""
result = evaluate_text_complexity(text=sample_text, grade_level=3)
pp.pprint(result["formatted_output"] if isinstance(result, dict) else result)

Calculated Flesch-Kincaid Score: 3.3
{'complexity_score': 'slightly_complex',
 'details': {'adjustment_and_scaffolding': [{'scaffolding_need': 'Connecting '
                                                                 'Mythology to '
                                                                 'Science',
                                             'suggestion': 'Provide a visual '
                                                           'aid or a simple '
                                                           'family tree '
                                                           'showing Apollo and '
                                                           'Artemis to help '
                                                           'students visualize '
                                                           "the 'twin' "
                                                           'relationship '
                                                           'mention

In [7]:
import pprint as pp
# I/O trace breakdown -- this is what the SDK engineer will replicate in TS.
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(result["rendered_prompt"])

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(result["formatted_output"])

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

RENDERED PROMPT (input sent to the LLM)
[{'additional_kwargs': {},
  'content': '**Role** You are an expert reading specialist and literacy '
             'evaluator. \n'
             '\n'
             '**Objective** Evaluate a provided text excerpt to determine its '
             'level of Intertextual complexity based on expert heuristic '
             'rules. You must carefully distinguish between self-contained '
             'information and information that requires familiarity with '
             'broader discourses, specialized domains, or specific external '
             'references.\n'
             '\n'
             '**Key Concepts & Definitions** \n'
             '**1. Separating Self-Contained Knowledge from Intertextuality & '
             'Specialized Domains (Step 0 Filter)** \n'
             'Intertextuality measures what prior textual, historical, '
             'cultural, or specialized domain encounters the text assumes and '
             'requires. \n'
             

In [8]:
# -------------------------------------------------------------------------
# Sniff-test runner: load fixtures.json and check predictions against expected
# -------------------------------------------------------------------------
# Fixtures live next to config.json + system.txt + user.txt and follow the
# schema declared in CONFIG['fixtures']['schema']. Each case has:
#   - id, description (optional)
#   - input: {text, grade_level}                  -- runtime evaluator inputs
#   - expected: {complexity_level}                -- ground-truth label from rubric
#
# Note: the fixture key 'complexity_level' maps to the runtime output's
# 'complexity_score' field. We test that single value only -- the model's
# free-text 'reasoning' field is non-deterministic across runs.

fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["sniff_test_path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

# Adjacency tolerance per CONFIG['fixtures']['tolerance'].
# Derive rubric order from OUTPUT_SCHEMA -- 'more_context_needed' excluded as it has no adjacency.
_RUBRIC_ORDER = [
    level for level in OUTPUT_SCHEMA["properties"]["complexity_score"]["enum"]
    if level != "more_context_needed"
]
_ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))

def _score_outcome(predicted: str, expected: str):
    """Return ('exact' | 'adjacent' | 'fail', distance_or_None)."""
    if predicted == expected:
        return "exact", 0
    if _ALLOW_ADJ and predicted in _RUBRIC_ORDER and expected in _RUBRIC_ORDER:
        d = abs(_RUBRIC_ORDER.index(predicted) - _RUBRIC_ORDER.index(expected))
        if d == 1:
            return "adjacent", d
    return "fail", None

# Run each fixture, accumulate results
results = []
for fx in fixtures:
    expected = fx["expected"]["complexity_level"]
    out = evaluate_text_complexity(
        text=fx["input"]["text"],
        grade_level=fx["input"]["grade_level"],
    )
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "predicted": None, "expected": expected, "error": out})
        continue
    predicted = out["formatted_output"]["complexity_score"]
    status, _ = _score_outcome(predicted, expected)
    results.append({
        "id": fx["id"], "status": status,
        "predicted": predicted, "expected": expected,
        "description": fx.get("description", ""),
    })

# Per-case report
print("\n" + "=" * 78)
print(f"{'ID':>5}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
print("=" * 78)
for r in results:
    icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
    print(f"{r['id']:>5}  {icon:<8}  {(r['predicted'] or '-'):<22}  {r['expected']:<22}  {r.get('description','')[:25]}")

# Summary
n_total = len(results)
n_exact = sum(1 for r in results if r["status"] == "exact")
n_adj   = sum(1 for r in results if r["status"] == "adjacent")
n_fail  = sum(1 for r in results if r["status"] == "fail")
n_err   = sum(1 for r in results if r["status"] == "error")
print("=" * 78)
print(f"Summary: {n_exact} exact, {n_adj} adjacent (tolerated), {n_fail} fail, {n_err} error  --  total {n_total}")
if _ALLOW_ADJ:
    print("(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)")
     

Loaded 4 fixtures from fixtures.json

Calculated Flesch-Kincaid Score: 3.3
Calculated Flesch-Kincaid Score: 4.12
Calculated Flesch-Kincaid Score: 7.33
Calculated Flesch-Kincaid Score: 10.24

   ID  STATUS    PREDICTED               EXPECTED                DESCRIPTION
NASA-001  PASS      slightly_complex        slightly_complex        What Is the Artemis Progr
NASA-074  PASS      moderately_complex      moderately_complex      What Is a Light-Year?
FYM-1106  PASS      moderately_complex      moderately_complex      A Good Night’s Sleep: Nec
CC-3983  PASS*     very_complex            exceedingly_complex     Italy's Violation of Fait
Summary: 3 exact, 1 adjacent (tolerated), 0 fail, 0 error  --  total 4
(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)
